# Transfer Learning

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/cnns/03-transfer-learning

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — don't start from scratch

Training a deep vision model from zero needs millions of images and huge compute. **Transfer
learning** sidesteps that: take a network already trained on a giant dataset (ImageNet), **reuse its
learned features**, and adapt it to your task. It works because a CNN's early layers learn *generic*
features (edges, textures, shapes) that transfer across almost any vision problem — only the final
task-specific head needs relearning. The two dials are **how much to freeze** (little data → freeze
the backbone, train only a new head; lots of data → fine-tune deeper) and **the learning rate**
(small for pretrained weights, larger for the new head). We work the parameter math exactly and show
the real `torchvision` pattern.

## Transfer Learning Strategies

Different strategies based on dataset size and similarity to source domain.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
strategies = [
    ('Small dataset\n+ Similar domain', 'Feature extraction\n(Freeze all layers)', '#818cf8'),
    ('Small dataset\n+ Different domain', 'Feature extraction\n+ New classifier', '#14b8a6'),
    ('Large dataset\n+ Similar domain', 'Fine-tune\nlast few layers', '#eab308'),
    ('Large dataset\n+ Different domain', 'Fine-tune\nall layers', '#f43f5e'),
]
for i, (label, strategy, color) in enumerate(strategies):
    ax.barh(i, 1, color=color, alpha=0.6, height=0.6)
    ax.text(0.5, i + 0.15, strategy, ha='center', va='center', color='white', fontsize=10)
    ax.text(-0.15, i, label, ha='right', va='center', color='#94a3b8', fontsize=10)
ax.set_xlim(-2.5, 1.5)
ax.axis('off')
ax.set_title('Transfer Learning Strategy Guide', color='white', fontsize=13)
plt.tight_layout()
plt.show()

**What to notice:** the three strategies form a spectrum — **feature extraction** (freeze
everything, train a new head) for small datasets, **fine-tune the last block** for medium, **fine-tune
everything** for large datasets similar to the source. Less data → freeze more (fewer parameters to
overfit); more data → adapt more of the network.

## Choosing a strategy by data size

| Data | Strategy | What to train |
|------|----------|---------------|
| Little | Feature extraction | Freeze backbone, train new head |
| Medium | Fine-tune top layers | Unfreeze last blocks + head |
| Lots | Full fine-tune | Train all layers (small LR) |

In [ ]:
# Sketch (PyTorch / torchvision). Freeze a pretrained backbone, swap the head.
try:
    import torch.nn as nn
    from torchvision import models
    net = models.resnet18(weights='IMAGENET1K_V1')
    for p in net.parameters():       # freeze backbone
        p.requires_grad = False
    net.fc = nn.Linear(net.fc.in_features, 5)   # new 5-class head (trainable)
    trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
    total = sum(p.numel() for p in net.parameters())
    print(f'training {trainable:,} of {total:,} params ({100*trainable/total:.1f}%)')
except ImportError:
    print('pip install torch torchvision to run this cell')

**What to notice:** the `torchvision` pattern is three lines — load a pretrained ResNet, set
`requires_grad = False` on the backbone (freeze it), and replace `net.fc` with a new head. Only the
head trains. (This cell runs in Colab, where `torch`/`torchvision` are preinstalled; locally it prints
an install hint.) Freezing means the backbone is a fixed **feature extractor**.

In [ ]:
# Pure-stdlib param-count math for ResNet-50 transfer learning.
# No torch/numpy needed -> fully deterministic and hand-checkable.

def linear_params(d_in, d_out):
    """Params in an FC layer: weights (d_in*d_out) + biases (d_out)."""
    return d_in * d_out + d_out

RESNET50_TOTAL = 25_557_032          # published torchvision count
orig_head = linear_params(2048, 1000)  # ImageNet head: Linear(2048, 1000)
backbone  = RESNET50_TOTAL - orig_head # everything except the head (frozen)

print(f"ResNet-50 total params : {RESNET50_TOTAL:,}")
print(f"Original head (2048->1000): {orig_head:,}")
print(f"Frozen backbone        : {backbone:,}")
print()

# --- Strategy 1a: two-layer head, k=2 classes (pneumonia vs normal) ---
h1 = linear_params(2048, 256)   # Linear(2048, 256)
h2 = linear_params(256, 2)      # Linear(256, 2); ReLU/Dropout have 0 params
seq_head = h1 + h2
new_total_seq = backbone + seq_head
print("Feature extraction with two-layer head (k=2):")
print(f"  Linear(2048->256) = {h1:,}")
print(f"  Linear(256->2)    = {h2:,}")
print(f"  new head total    = {seq_head:,}")
print(f"  trainable / total = {seq_head:,} / {new_total_seq:,} "
      f"({100*seq_head/new_total_seq:.3f}%)")
print()

# --- Strategy 1b: plain Linear(2048, 2) head ---
simple_head = linear_params(2048, 2)
new_total_simple = backbone + simple_head
print("Feature extraction with plain Linear(2048->2) head:")
print(f"  trainable / total = {simple_head:,} / {new_total_simple:,} "
      f"({100*simple_head/new_total_simple:.4f}%)")
print()

# --- Strategy 2: fine-tune layer4 + head ---
LAYER4 = 14_964_736  # published ResNet-50 layer4 param count
ft_trainable = LAYER4 + seq_head
print("Fine-tuning layer4 + new head:")
print(f"  layer4 alone        = {LAYER4:,} ({100*LAYER4/RESNET50_TOTAL:.1f}% of net)")
print(f"  trainable / total   = {ft_trainable:,} / {new_total_seq:,} "
      f"({100*ft_trainable/new_total_seq:.1f}%)")

# --- Cross-check / verification against the values quoted in the lesson ---
assert orig_head == 2_049_000
assert backbone == 23_508_032
assert seq_head == 525_058
assert new_total_seq == 24_033_090
assert simple_head == 4_098
assert abs(100*seq_head/new_total_seq - 2.18) < 0.01      # ~2.2%
assert abs(100*simple_head/new_total_simple - 0.0174) < 0.001  # ~0.017%
assert abs(100*LAYER4/RESNET50_TOTAL - 58.6) < 0.1        # ~58.6%
assert abs(100*ft_trainable/new_total_seq - 64.4) < 0.1   # ~64%
print("\nAll lesson numbers verified.")

**What to notice:** the parameter arithmetic is exact and hand-checkable. **Feature extraction**
(swap `Linear(2048,1000)` for `Linear(2048,2)`) trains just **~0.016%** of ResNet-50's 25.6M
parameters — essentially free. **Fine-tuning `layer4`** retrains ~15M params (**~59%**) — much more
capacity, needing more data to avoid overfitting. This math is *why* the freeze-more-with-less-data
rule holds.

## Discriminative learning rates

When fine-tuning, deeper (earlier) layers get a **smaller** learning rate than the head. A common rule is a geometric decay across $L$ layer groups:

$$
\eta_\ell = \eta_{\text{head}}\cdot \gamma^{\,(L-\ell)},\qquad \gamma\in(0,1).
$$

With $\eta_{\text{head}} = 10^{-3}$, $\gamma = 0.3$ and $L = 5$ groups (group 5 = head, group 1 = earliest), the schedule is computed below — again in pure stdlib so it is exact.

In [ ]:
# Discriminative (layer-wise) learning-rate schedule -- pure stdlib, deterministic.
eta_head = 1e-3
gamma = 0.3
L = 5  # 5 groups: group 5 = head (gets eta_head), group 1 = earliest layers

print(f"{'group':>6} {'role':<12} {'learning rate':>14}")
schedule = []
for layer in range(L, 0, -1):  # 5,4,3,2,1
    lr = eta_head * gamma ** (L - layer)
    schedule.append(lr)
    role = 'head' if layer == L else f'block {layer}'
    print(f"{layer:>6} {role:<12} {lr:>14.2e}")

# Cross-check a couple of values by hand:
#   group 5 (head): 1e-3 * 0.3**0 = 1e-3
#   group 4:        1e-3 * 0.3**1 = 3e-4
#   group 1:        1e-3 * 0.3**4 = 8.1e-6
assert abs(schedule[0] - 1e-3) < 1e-12
assert abs(schedule[1] - 3e-4) < 1e-12
assert abs(schedule[-1] - 1e-3 * 0.3**4) < 1e-12
assert abs(schedule[-1] - 8.1e-6) < 1e-9
# The head LR is ~123x the earliest-block LR:
print(f"\nhead LR / earliest-block LR = {schedule[0]/schedule[-1]:.1f}x")
assert abs(schedule[0]/schedule[-1] - 0.3**-4) < 1e-6  # = 1/0.3^4
print("Discriminative-LR schedule verified.")

**What to notice:** **discriminative learning rates** apply a *smaller* LR to early (pretrained,
general) layers and a *larger* LR to later (task-specific) layers — the schedule decays geometrically
from head to backbone. This lets the head adapt quickly while barely disturbing the valuable
pretrained features.

## Gotchas & tradeoffs

- **Catastrophic forgetting.** Too large a learning rate on the pretrained weights wipes out the
  features you're trying to reuse — always fine-tune with a small LR (÷10–100 vs from-scratch).
- **Domain gap.** ImageNet features transfer well to natural images but less to X-rays, satellite, or
  audio spectrograms — the further the domain, the more layers you must fine-tune.
- **Frozen BatchNorm.** A frozen backbone should run its BatchNorm in **eval mode** (use the running
  stats), or it silently keeps adapting to your small batch and hurts performance.
- **Negative transfer.** If the source task is unrelated, pretrained features can *hurt* — transfer
  isn't always a win.

In [ ]:
# The freeze-vs-finetune tradeoff, in exact parameter counts
RESNET50_TOTAL = 25_557_032
def frac(trained): return 100 * trained / RESNET50_TOTAL

feature_extraction = linear_params(2048, 2)          # swap head only
finetune_layer4    = 14_964_736                      # ResNet-50 layer4 (~15M)
print(f'feature extraction (new head only): {feature_extraction:>10,} params  ({frac(feature_extraction):.3f}%)')
print(f'fine-tune layer4                  : {finetune_layer4:>10,} params  ({frac(finetune_layer4):.1f}%)')
print(f'fine-tune everything              : {RESNET50_TOTAL:>10,} params  (100.0%)')
assert frac(feature_extraction) < 0.1, "feature extraction trains a tiny fraction"
print('\n-> less data -> train fewer params (freeze more); more data -> fine-tune deeper')

**What to notice:** the three strategies span **0.017% → 59% → 100%** of the parameters. Feature
extraction is almost free and needs little data; full fine-tuning is powerful but data-hungry and
risks forgetting. Choosing where on this spectrum to sit *is* the transfer-learning decision, and the
parameter counts make the tradeoff concrete.

## Key takeaways

- **Transfer learning** reuses features learned on a huge dataset (ImageNet) for your task.
- **Freezing** discards a layer's gradient so its weights never update — the backbone becomes a fixed feature extractor and only the head learns.
- A swapped head is tiny: ResNet-50's `Linear(2048, 1000)` ImageNet head (2,049,000 params) becomes `Linear(2048, 2)` (4,098 params); feature extraction trains as little as **0.017%** of the network.
- "Fine-tune the last block" retrains the *majority* of weights: ResNet-50's `layer4` is ~15M params (**58.6%** of the model).
- Use **discriminative learning rates** (small for the pretrained backbone, larger for the new head) and a from-scratch-÷10–100 LR to avoid catastrophic forgetting.
- It slashes data needs and training time versus training from scratch.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — Discriminative learning rates

Implement the geometric schedule from the section above:

$$\eta_\ell = \eta_{\text{head}} \cdot \gamma^{(L - \ell)}, \qquad \gamma \in (0, 1)$$

Group $L$ is the head (full base LR); each earlier group is scaled down by another factor of $\gamma$ — general features need gentle nudges, the new head needs real training. The checks reproduce the exact values printed above.

In [ ]:
def layer_lr(eta_head, gamma, L, layer):
    """Learning rate for group `layer` (1 = earliest, L = head)."""
    # TODO(you): eta_head * gamma ** (L - layer)
    return ...

In [ ]:
# Checks — run me
assert abs(layer_lr(1e-3, 0.3, 5, 5) - 1e-3) < 1e-15, "the head gets the full base LR"
assert abs(layer_lr(1e-3, 0.3, 5, 4) - 3e-4) < 1e-15, "one group down: x0.3"
assert abs(layer_lr(1e-3, 0.3, 5, 1) - 8.1e-6) < 1e-12, "earliest group: 1e-3 * 0.3^4"

lrs = [layer_lr(1e-3, 0.3, 5, l) for l in range(1, 6)]
assert all(a < b for a, b in zip(lrs, lrs[1:])), "earlier layers always get smaller LRs"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def layer_lr(eta_head, gamma, L, layer):
    return eta_head * gamma ** (L - layer)
```

</details>

### Exercise 2 — What fraction do you actually train?

Feature extraction freezes the backbone and trains only a new head. Build the head's parameter count from its layer sizes (each $d_{in} \to d_{out}$ linear layer costs $d_{in} d_{out} + d_{out}$), then compute the trainable fraction. With ResNet-50's ~25M-parameter backbone, the punchline of the checks: you train **about 2%** of the model.

In [ ]:
def linear_params(d_in, d_out):
    return d_in * d_out + d_out


def head_params(sizes):
    """Total params of a linear head with the given layer sizes."""
    # TODO(you): sum linear_params over consecutive pairs of sizes
    return ...


def trainable_fraction(backbone_params, head_sizes):
    """Fraction of the whole model that is trainable when the backbone is frozen."""
    head = head_params(head_sizes)

    # TODO(you): head / (backbone + head)
    return ...

In [ ]:
# Checks — run me
assert head_params([2048, 2]) == 4098, "single-layer head: 2048*2 + 2"
assert head_params([2048, 256, 2]) == 525058, "two-layer head: (2048*256+256) + (256*2+2)"

RESNET50_BACKBONE = 25_557_032 - linear_params(2048, 1000)   # total minus the ImageNet head
frac = trainable_fraction(RESNET50_BACKBONE, [2048, 256, 2])
assert frac < 0.025, "with a frozen ResNet-50 backbone, you train ~2% of the parameters"
assert trainable_fraction(RESNET50_BACKBONE, [2048, 2]) < frac, "a smaller head trains even less"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def head_params(sizes):
    return sum(linear_params(a, b) for a, b in zip(sizes[:-1], sizes[1:]))


def trainable_fraction(backbone_params, head_sizes):
    head = head_params(head_sizes)
    return head / (backbone_params + head)
```

</details>